# Q²-JEPA on Colab GPU

Thin driver: all code lives in the `q2jepa` package. Runtime → Change runtime type → **GPU**.
Re-running the training cell after a disconnect resumes from the Drive checkpoint.

In [ ]:
!nvidia-smi
from pathlib import Path

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

In [ ]:
REPO_URL = "https://github.com/Brigada-Inginerilor-Amarati/Q2-JEPA"
!git clone -q {REPO_URL} /content/Q2-JEPA
%cd /content/Q2-JEPA
!pip install -q -e .
!python scripts/colab_setup.py
import site

site.main()

## 1. Quantum kernels on the `nvidia` target

In [ ]:
!Q2JEPA_CUDAQ_TARGET=nvidia python -m pytest tests/test_born.py tests/test_attention_equivalence.py -q

## 2. Swap-test attention benchmark: CPU vs GPU

In [ ]:
!python scripts/bench_attention.py --target qpp-cpu --res 16 --res 32 --images 16
!python scripts/bench_attention.py --target nvidia --res 16 --res 32 --images 16
!nvidia-smi --query-gpu=name --format=csv

## 3. Training run (checkpoints to Drive; re-run this cell to resume)

In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_dir

import q2jepa.config  # noqa: F401
from q2jepa.train import run

OVERRIDES = ["quantum=gpu", "trainer.device=cuda", "encoder.res=64", "seed=0"]
OUT = Path("/content/drive/MyDrive/q2jepa/runs/classical-res64-s0")
with initialize_config_dir(config_dir="/content/Q2-JEPA/conf", version_base="1.3"):
    cfg = compose("config", overrides=OVERRIDES)
run(cfg, OUT)